In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 2 - WEEK 10 BAYESIAN OPTIMISATION
# Run from inside the week10/ folder
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 9.
# - Check Week 9 realised calibration.
# - Search local + wider + global candidate pools.
# - Keep global EI as a diagnostic.
# - Compare EI, posterior mean and several UCB settings.
# - Do NOT automatically accept uncertainty-driven global moves.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 10 cumulative data
# ------------------------------------------------------------

X = np.load("function2/initial_inputs.npy")
Y = np.load("function2/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. WEEK 9 CALIBRATION CHECK
# ------------------------------------------------------------
#
# Week 9 selected:
# [0.704000, 0.982000]
#
# Week 9 GP prediction for beta = 0.5 candidate:
# mean ≈ 0.611199
# std  ≈ 0.066595
#
# Actual Week 9 output:
# 0.6175014783909925
# ------------------------------------------------------------

week9_pred_mean = 0.611199
week9_pred_std = 0.066595
week9_actual = 0.6175014783909925

week9_error = (
    week9_actual
    - week9_pred_mean
)

week9_z_error = (
    week9_error
    / week9_pred_std
)

print("\n================================")
print("WEEK 9 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week9_pred_mean)
print("Predicted std :", week9_pred_std)
print("Actual        :", week9_actual)

print("\nPrediction error:")
print(week9_error)

print("\nError / predicted std:")
print(week9_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------
#
# F2 has repeatedly shown a strong region around:
#
# x1 ~ 0.70
# x2 high / near 1
#
# But we retain a large global pool to detect whether the
# updated GP has found genuinely compelling evidence elsewhere.
#
# Local/wide widths are derived automatically from ARD.
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


# Local search around ACTUAL incumbent

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(100000, 2)
    )
)


# Wider search around ACTUAL incumbent

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(80000, 2)
    )
)


# Full-domain coverage

global_candidates = rng.uniform(
    0,
    1,
    size=(150000, 2)
)


# Explicit dense boundary coverage for x2 = 1
#
# Since the actual incumbent lies exactly on the x2 boundary,
# include dense coverage there rather than relying only on
# clipped random samples.

boundary_x1 = np.linspace(
    0,
    1,
    5001
)

boundary_candidates = np.column_stack([
    boundary_x1,
    np.ones_like(boundary_x1)
])


# Clip stochastic pools to domain

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)


# Combine

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates,
    boundary_candidates
])


# ------------------------------------------------------------
# 6. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 7. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 8. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 9. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 10. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance-from-incumbent diagnostic
# ------------------------------------------------------------
#
# Helps distinguish a genuine local refinement from a large
# uncertainty-driven jump elsewhere in the domain.
# ------------------------------------------------------------

def distance_from_best(x):
    return np.linalg.norm(
        x - best_x
    )


print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    distance_from_best(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    distance_from_best(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        distance_from_best(
            candidates[idx]
        )
    )


# ------------------------------------------------------------
# 13. x2 boundary diagnostic
# ------------------------------------------------------------

print("\n================================")
print("X2 BOUNDARY CHECK")
print("================================")

print(
    "EI x2:",
    candidates[ei_idx, 1]
)

print(
    "Highest mean x2:",
    candidates[mean_idx, 1]
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta} x2:",
        candidates[idx, 1]
    )

DATA
X shape: (19, 2)
Y shape: (19,)

Current best:
[0.711177 1.      ] -> 0.679662733033218

Y range:
min = -0.06562362443733738
max = 0.679662733033218
std = 0.24726984015840672

WEEK 9 CALIBRATION CHECK
Predicted mean: 0.611199
Predicted std : 0.066595
Actual        : 0.6175014783909925

Prediction error:
0.006302478390992405

Error / predicted std:
0.09463891269603432


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.03**2 * Matern(length_scale=[0.0585, 2], nu=2.5) + WhiteKernel(noise_level=0.0466)

ARD lengthscales:
[0.05851642 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.9715735 0.0284265]

CANDIDATE SCALES
Local widths:
[0.015 0.1  ]

Wide widths:
[0.04 0.2 ]

Candidates after duplicate filtering:
283890

PRIMARY EI
candidate = [0.96472083 0.00274275]
mean = 0.3775749006428918
std = 0.24997140599353443
EI = 0.013781557235878573

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [0.96472083 0.00274275] 
 mean = 0.377575 
 std = 0.249971 
 EI = 0.01378156 

xi = 2.472698e-03 
 candidate = [0.96472083 0.00274275] 
 mean = 0.377575 
 std = 0.249971 
 EI = 0.01350342 

xi = 1.236349e-02 
 candidate = [0.96472083 0.00274275] 
 mean = 0.377575 
 std = 0.249971 
 EI = 0.01243678 

xi = 2.472698e-02 
 candidate = [0.96810639 0.00350197] 
 mean = 0.374934 
 std = 0.251485 
 EI = 0.01120728 


HIGHEST PREDICTED MEAN
candidate = [0.70504104 0.97114282]
mean = 0.6138332

In [2]:
# ============================================================
# FINAL FUNCTION 2 - WEEK 10 SELECTION
# ============================================================
#
# Week 9 calibration was excellent:
# realised error ≈ +0.095 predictive standard deviations.
#
# Global EI is rejected because it jumps to a distant,
# high-uncertainty region with substantially lower mean.
#
# Posterior mean and all UCB settings remain in the established
# high-performing basin around x1 ~ 0.70 and high x2.
#
# beta = 1.0 is selected because, in this case, it moves the
# candidate CLOSER to the proven incumbent at x2 = 1 rather
# than toward an unsupported exploratory region.

beta = 1.0

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week10_candidate = candidates[final_idx]

print("Week 10 Function 2 candidate:")
print(week10_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week10_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week10_candidate
)

print("\nPortal format:")
print(portal)

Week 10 Function 2 candidate:
[0.70231586 0.99507643]

Predicted mean:
0.6126483620009504

Predicted std:
0.06085811464527108

UCB:
0.6735064766462215

Distance from current best:
0.010137125752629914

Portal format:
0.702316-0.995076
